# 09 · Od perceptronu k vyladěnému klasifikátoru

Praktická část cvičení 09. Na datech z kontroly výlisků naučíte perceptron a logistický neuron,
vyhodnotíte je metrikami z přednášky, **samostatně je vyladíte**, zvolíte rozhodovací práh podle ceny chyb
a výsledek jednou otestujete. Každé nastavení, které zde ladíte, jste viděli jako graf v edukační části `perceptron-visual-cz.ipynb`.

Krátké spustitelné ukázky používají malá ilustrační data a proměnné s předponou `demo_`. Spouštějte je v pořadí:
na sebe navazují a ukazují syntaxi celého postupu. Vlastní řešení pište do prázdných kódových buněk pod úkoly.
Testovací data se otevřou až v kroku 7. Po dokončení spusťte `Restart Kernel and Run All` a ověřte, že řešení běží v pořadí.

## Příprava prostředí a načtení dat

1. Otevřete notebook v Jupyteru (balíčky popisuje README) nebo v Google Colab.
2. Spusťte ukázku níže. Importuje všechny potřebné nástroje a připraví malá ilustrační data `demo_`.
3. Připravená buňka pod ní načte data do tabulky `raw` a nastaví `SEED = 42` pro všechny náhodné kroky. Když notebook běží ve složce
   cvičení nebo v kořeni repozitáře, použije místní soubor `data/parts.csv`; jinak stejný soubor stáhne z GitHubu
   (je potřeba připojení k internetu).

### Ukázka: importy a ilustrační data

In [ ]:
import warnings                                              # Potlačení varování o nedokončeném učení (viz níže).
from IPython.display import display                          # Zobrazení tabulek ve vlastních úkolech.
import matplotlib.pyplot as plt                              # Grafy.
import numpy as np                                           # Numerické výpočty.
import pandas as pd                                          # Data a tabulky výsledků.
from sklearn.base import clone                               # Nová nenaučená kopie nastavení modelu.
from sklearn.model_selection import train_test_split         # Rozdělení vstupů a cílů najednou.
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict  # Mřížkové hledání a křížová validace.
from sklearn.pipeline import Pipeline                        # Řetězení kroků do jednoho modelu.
from sklearn.preprocessing import StandardScaler, PolynomialFeatures  # Standardizace a nové součinové příznaky.
from sklearn.linear_model import Perceptron, SGDClassifier  # Perceptron a neuron učený stochastickým gradientním sestupem.
from sklearn.dummy import DummyClassifier                    # Jednoduchý referenční model.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Matice záměn jako čísla a jako graf.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score  # Metriky tvrdých rozhodnutí.
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay  # Metriky a křivky přes všechny prahy.
from sklearn.exceptions import ConvergenceWarning            # Typ varování pro učení zastavené limitem epoch.

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # Malé rychlosti a málo epoch ladíme záměrně; posuzujeme výsledek, ne varování.

demo_rng = np.random.default_rng(7)                          # Ilustrační data, ne data úlohy.
demo_X = pd.DataFrame({"size": demo_rng.normal(50, 10, 300),       # Příznak v desítkách.
                       "weight": demo_rng.normal(2000, 300, 300)})  # Příznak v tisících.
demo_risk = -2.2 + 0.09 * (demo_X["size"] - 50) + 0.004 * (demo_X["weight"] - 2000)  # Skryté riziko třídy 1.
demo_y = (demo_rng.random(300) < 1 / (1 + np.exp(-demo_risk))).astype(int)  # Asi 15 % třídy 1.
demo_X_dev, demo_X_test, demo_y_dev, demo_y_test = train_test_split(demo_X, demo_y, test_size=0.25, stratify=demo_y, random_state=7)  # Čtvrtina pro test.
demo_X_train, demo_X_val, demo_y_train, demo_y_val = train_test_split(demo_X_dev, demo_y_dev, test_size=0.25, stratify=demo_y_dev, random_state=7)  # Čtvrtina zbytku pro validaci.
print(len(demo_X_train), len(demo_X_val), len(demo_X_test), "podíl třídy 1:", round(demo_y.mean(), 3))  # Velikosti tří částí.

**Připravený kód:**

In [ ]:
from pathlib import Path                                     # Cesty k místní kopii dat.
SEED = 42                                                    # Stejné náhodné dělení a učení pro všechny.
DATA_URL = ("https://raw.githubusercontent.com/CVUT-FS-12110/Artificial-Intelligence-and-Neural-Networks/"
            "master/09-Perceptron/data/parts.csv")           # Tentýž soubor v repozitáři předmětu na GitHubu.
local = [Path("data/parts.csv"), Path("09-Perceptron/data/parts.csv")]  # Složka cvičení nebo kořen repozitáře.
found = [path for path in local if path.exists()]           # Místní kopie, které existují.
source = found[0] if found else DATA_URL                     # Bez místní kopie (např. Google Colab) stáhnout z GitHubu.
raw = pd.read_csv(source)                                    # 2000 zkontrolovaných výlisků.
print("data načtena z:", source)          # Místní soubor nebo adresa URL.
display(raw.head())  # Prvních pět řádků.

## Úloha a data

Každý řádek je jeden **vstřikovaný výlisek** s osmi nastaveními procesu změřenými při jeho výrobě a výsledkem
výstupní kontroly `defect` (1 = vadný). Data jsou syntetická; jejich popis je v `data/README.md`.

| Sloupec | Význam | Typická hodnota |
|---|---|---|
| `melt_temp` | teplota taveniny | 230 °C |
| `pressure` | vstřikovací tlak | 900 bar |
| `cooling_time` | doba chlazení | 12 s |
| `mould_temp` | teplota formy | 60 °C |
| `humidity` | vlhkost ve skladu granulátu | 45 % |
| `vibration` | vibrace lisu | 0,5 mm/s |
| `shot_weight` | hmotnost dávky | 52 g |
| `cycle_time` | doba cyklu | 30 s |
| `shrinkage` | naměřené smrštění – v tomto cvičení se nepoužívá, **není vstupem** | 1,5 % |

**Cíl:** předpovědět výsledek kontroly z nastavení, aby se podezřelé výlisky poslaly na podrobnou kontrolu.
**Ceny:** vadný výlisek odeslaný zákazníkovi (FN) stojí **10** jednotek (reklamace), dobrý výlisek poslaný na podrobnou kontrolu
(FP) stojí **1** jednotku. Kvalita rozhodnutí se proto měří **celkovou cenou** C = 10·FN + 1·FP,
ne samotnou správností.

## 1. Prohlídka dat (připraveno)

**Výstup:** podíl vadných výlisků a tabulka rozsahů příznaků. Všimněte si, jak vzácná je třída 1 a jak rozdílná jsou měřítka.

**Připravený kód:**

In [ ]:
print("řádků:", len(raw), "| chybějících hodnot:", int(raw.isna().sum().sum()))  # Velikost a úplnost.
print("podíl vadných výlisků:", round(raw["defect"].mean(), 3))  # Podíl třídy 1.
display(raw.drop(columns=["part_id", "defect"]).describe().T[["mean", "std", "min", "max"]].round(3))  # Rozsahy vstupů.

## 2. Příznaky a pevné rozdělení (připraveno)

**Výstup:** `X` (osm vstupů) a `y` (`defect`); vývojová část `X_dev`, `y_dev` (1500 výlisků) a testovací část `X_test`, `y_test`
(500 výlisků), která zůstává zavřená až do kroku 7. Vývojová část se dál dělí na trénovací `X_train`, `y_train` (1125)
a validační `X_val`, `y_val` (375). Všechna dělení jsou stratifikovaná: každá část zachovává podíl vadných.

**Připravený kód:**

In [ ]:
FEATURES = ["melt_temp", "pressure", "cooling_time", "mould_temp", "humidity", "vibration", "shot_weight", "cycle_time"]  # Osm vstupů.
X = raw[FEATURES]                                            # Osm vstupů; part_id a shrinkage nejsou vstupy.
y = raw["defect"]                                            # Cílová třída.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)  # Čtvrtina pro test, stratifikovaně.
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)  # Čtvrtina zbytku pro validaci.
for name, part in (("train", y_train), ("validation", y_val), ("dev", y_dev), ("test", y_test)):  # Výlisky a vady v každé části.
    print(f"{name:10s} {len(part):5d} výlisků, {int(part.sum()):4d} vadných")  # Jeden řádek na část.

## 3. Referenční model a funkce pro vyhodnocení (připraveno)

**Výstup:** funkce `evaluate(y_true, y_pred)`, která vrací počty z matice záměn, pět metrik a celkovou cenu,
a tabulka `reference` pro model, který vždy předpoví nejčastější třídu („každý výlisek je dobrý“).
Funkci `evaluate` použijte ve všech dalších krocích, aby byly výsledky srovnatelné.

**Připravený kód:**

In [ ]:
C_FN, C_FP = 10, 1                                           # Cena přehlédnuté vady a planého poplachu.


def evaluate(y_true, y_pred):  # Používá se ve všech dalších krocích.
    # Počty z matice záměn, metriky a celková cena tvrdých rozhodnutí 0/1.
    y_pred = np.asarray(y_pred).astype(int)                  # Přijme i True/False.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()  # Čtyři počty v tomto pořadí.
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),  # Počty z matice záměn.
            "správnost": accuracy_score(y_true, y_pred),  # (TP + TN) / n
            "přesnost": precision_score(y_true, y_pred, zero_division=0),  # 0, když není žádný poplach
            "úplnost": recall_score(y_true, y_pred),  # TP / (TP + FN)
            "F1": f1_score(y_true, y_pred, zero_division=0),  # harmonický průměr přesnosti a úplnosti
            "vyvážená správnost": balanced_accuracy_score(y_true, y_pred),  # (úplnost + specificita) / 2
            "cena": int(C_FN * fn + C_FP * fp)}  # C = 10·FN + 1·FP


baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)  # Vždy předpoví „dobrý“.
reference = pd.DataFrame([evaluate(y_val, baseline.predict(X_val))], index=["vždy dobrý"])  # Validační výsledek reference.
display(reference.round(3))  # Zaokrouhlená tabulka.

**Zamyslete se:** Referenční model má správnost asi 0,89. Proč je přesto k ničemu? Které metriky a jaká cena to ukazují?

## 4. Perceptron

**Cíl / výstup:** naučený `perceptron_pipe`, jeho validační předpovědi `val_pred_perceptron`, matice záměn, tabulka
`perceptron_results` srovnávající perceptron se škálováním a bez něj a seznam `epoch_errors` s trénovacími chybami po každé ze 30 epoch.

1. `perceptron_pipe` = `Pipeline` s kroky `scale`: `StandardScaler()` → `model`: `Perceptron(random_state=SEED)`.
   Naučte ho na trénovacích datech a předpovězte validační data.
2. Nakreslete matici záměn validačních předpovědí. Ze čtyř počtů spočtěte **ručně** přesnost a úplnost
   a porovnejte je s `evaluate`.
3. Naučte stejný perceptron **bez** škálování (jen `Perceptron(random_state=SEED)`). Oba výsledky `evaluate`
   vložte do `perceptron_results` spolu s `reference`.
4. Trénovací chyby po epochách: naučte `StandardScaler` na `X_train`, vytvořte nový `Perceptron(random_state=SEED)` a zavolejte
   30krát `partial_fit` na škálovaných trénovacích datech (jedno volání = jedna epocha). Po každé epoše uložte počet trénovacích chyb
   do `epoch_errors` a seznam vykreslete.

**Použijte:** `Pipeline`, `StandardScaler`, `Perceptron`, `.fit`, `.predict`, `ConfusionMatrixDisplay.from_predictions`,
`confusion_matrix(...).ravel()`, `evaluate`, `pd.DataFrame`, `.partial_fit(X, y, classes=[0, 1])`, `plt.plot`.

<details><summary>Nápověda</summary>

`tn, fp, fn, tp = confusion_matrix(y_val, val_pred).ravel()` dává čtyři počty v tomto pořadí. `Perceptron` se učí
pravidlem z přednášky (oprava po každém chybně zařazeném vzorku, v každé epoše v náhodném pořadí) a skončí, když trénovací chyba
přestane klesat. `partial_fit` udělá právě jeden průchod daty a pokračuje z aktuálních vah; první volání potřebuje
seznam všech tříd. Škálovač se učí jen na trénovacích datech a pak je transformuje:
`scaler.transform(X_train)`.

</details>

**Zamyslete se:** Ustálí se počet trénovacích chyb s přibývajícími epochami? Co to říká o datech
(edukační část, oddíl 3)? Která metrika se mezi perceptronem se škálováním a bez něj liší nejvíc?

### Ukázka: perceptron, matice záměn a chyby po epochách

In [ ]:
demo_pipe = Pipeline([("scale", StandardScaler()), ("model", Perceptron(random_state=7))])  # Škálování, potom perceptron.
demo_pipe.fit(demo_X_train, demo_y_train)                    # Škálovač i váhy se učí jen z trénovacích dat.
demo_pred = demo_pipe.predict(demo_X_val)  # Validační předpovědi 0/1.
ConfusionMatrixDisplay.from_predictions(demo_y_val, demo_pred, colorbar=False)  # Řádky: skutečná třída, sloupce: předpověď.
plt.show()  # Zobrazit graf.
demo_tn, demo_fp, demo_fn, demo_tp = confusion_matrix(demo_y_val, demo_pred).ravel()  # Čtyři počty.
print("úplnost ručně:", demo_tp / (demo_tp + demo_fn), "| evaluate:", evaluate(demo_y_val, demo_pred)["úplnost"])  # Stejná hodnota oběma způsoby.

demo_scaler = StandardScaler().fit(demo_X_train)             # Samostatně naučený škálovač pro smyčku po epochách.
demo_Z = demo_scaler.transform(demo_X_train)  # Škálované trénovací vstupy.
demo_model, demo_errors = Perceptron(random_state=7), []  # Nový perceptron a prázdný seznam.
for demo_epoch in range(10):                                 # Deset epoch; jedno volání partial_fit = jeden průchod daty.
    demo_model.partial_fit(demo_Z, demo_y_train, classes=[0, 1])  # Jedna epocha.
    demo_errors.append(int((demo_model.predict(demo_Z) != demo_y_train).sum()))  # Trénovací chyby po epoše.
print("trénovací chyby po epochách:", demo_errors)  # Nemusí klesat.

**Vlastní kód:**

## 5. Logistický neuron

**Cíl / výstup:** naučený `logistic_pipe`, validační pravděpodobnosti třídy 1 `val_proba`, tabulka `step5_results`
s perceptronem a logistickým neuronem při prahu 0,5, čísla `val_auc` a `val_ap` a tři grafy:
křivky ROC a křivky přesnost–úplnost obou modelů a histogram `val_proba` pro každou třídu.

1. `logistic_pipe` = `Pipeline`: `scale`: `StandardScaler()` → `model`:
   `SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, alpha=1e-4, random_state=SEED)`.
   Je to jeden logistický neuron učený stochastickým gradientním sestupem s konstantní rychlostí učení η = `eta0`
   a malou regularizací vah `alpha`.
2. Naučte ho na trénovacích datech, uložte `val_proba = logistic_pipe.predict_proba(X_val)[:, 1]` a tvrdé předpovědi při 0,5 (`.predict`).
   Výsledek `evaluate` přidejte do `step5_results` spolu s perceptronem se škálováním z kroku 4.
3. Z `val_proba` spočtěte `val_auc` (ROC AUC) a `val_ap` (plocha pod křivkou přesnost–úplnost). Pro perceptron použijte jeho skóre
   `perceptron_pipe.decision_function(X_val)`: ROC a PR potřebují jen pořadí, ne pravděpodobnost.
4. Nakreslete obě křivky ROC do jednoho grafu a obě křivky přesnost–úplnost do druhého. Nakreslete histogram `val_proba` zvlášť pro
   dobré a vadné výlisky.

**Použijte:** `SGDClassifier`, `.predict_proba`, `.decision_function`, `roc_auc_score`, `average_precision_score`,
`RocCurveDisplay.from_predictions`, `PrecisionRecallDisplay.from_predictions`, `plt.hist`.

<details><summary>Nápověda</summary>

Bez `learning_rate="constant"` používá klasifikátor vlastní rozvrh a **`eta0` ignoruje** – ladění v kroku 6 by pak
nic neměnilo. `predict_proba` vrací dva sloupce (třída 0 a třída 1); `[:, 1]` vybere třídu 1. Obě zobrazovací funkce přijímají
`ax=` pro kreslení do připravené osy: `fig, axs = plt.subplots(1, 2)`, potom `RocCurveDisplay.from_predictions(y_val, scores, name="...", ax=axs[0])`.
Histogram jedné třídy: `plt.hist(val_proba[y_val == 1], bins=20)`.

</details>

**Zamyslete se:** AUC je zřetelně nad 0,5, a přesto je při prahu 0,5 úplnost nízká. Kde v histogramu leží většina vadných výlisků?
Co by se stalo s úplností, přesností a cenou při prahu 0,1?

### Ukázka: pravděpodobnosti, AUC a křivky

In [ ]:
demo_logistic = Pipeline([("scale", StandardScaler()),  # Nejdřív škálování …
                          ("model", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.05, random_state=7))])  # Jeden logistický neuron.
demo_logistic.fit(demo_X_train, demo_y_train)  # Učení na trénovacích datech.
demo_proba = demo_logistic.predict_proba(demo_X_val)[:, 1]  # Pravděpodobnost třídy 1.
print("AUC:", round(roc_auc_score(demo_y_val, demo_proba), 3), "| AP:", round(average_precision_score(demo_y_val, demo_proba), 3))  # Obě míry využívají celé pořadí.
fig, axs = plt.subplots(1, 3, figsize=(14, 4))  # Tři grafy vedle sebe.
RocCurveDisplay.from_predictions(demo_y_val, demo_proba, name="logistický", ax=axs[0], plot_chance_level=True)  # Křivka ROC s úhlopříčkou náhodného modelu.
PrecisionRecallDisplay.from_predictions(demo_y_val, demo_proba, name="logistický", ax=axs[1])  # Křivka přesnost–úplnost.
axs[2].hist(demo_proba[demo_y_val == 0], bins=10, alpha=0.6, label="y = 0")  # Pravděpodobnosti třídy 0 …
axs[2].hist(demo_proba[demo_y_val == 1], bins=10, alpha=0.6, label="y = 1")  # … a třídy 1.
axs[2].set(xlabel="pravděpodobnost třídy 1")  # Popis osy.
axs[2].legend()  # Legenda.
plt.tight_layout()  # Popisky se nepřekrývají.
plt.show()  # Zobrazit graf.

**Vlastní kód:**

## 6. Samostatné ladění

V kroku 5 jste použili nastavení dané zadáním. Teď ho **vylaďte sami**. Předzpracování a neuron se ladí společně.
Ladění má tři části:

- **6.1 Ruční ladění** na pevné validační části: několik variant, každá mění jednu věc. Rychlé a názorné, ale
  375 validačních výlisků obsahuje jen 40 vadných, takže rozdíly jsou zatížené šumem.
- **6.2 Mřížkové hledání** s pětinásobnou křížovou validací na vývojových datech: systematické a spolehlivější.
- **6.3 Rozhodovací práh** zvolený podle ceny chyb.

Metrikou ladění v 6.1 a 6.2 je **ROC AUC**. Nezávisí na prahu, a proto posuzuje, jak dobře model výlisky seřadí;
práh se volí zvlášť v 6.3. **Testovací data se k ladění nepoužívají.**

| Parametr | Možné hodnoty | Souvislost s edukační částí |
|---|---|---|
| `scale` | `"passthrough"`, `StandardScaler()` | 4 škálování vstupů, 5 rychlost učení |
| `features` | `"passthrough"`, `PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)` | 10 XOR a součin x₁·x₂ |
| `model__loss` | `"perceptron"` (perceptronové pravidlo), `"log_loss"` (logistický neuron) | 1 a 6 |
| `model__eta0` | 0,0001, 0,001, 0,01, 0,1, 1 | 5 rychlost učení |
| `model__alpha` | 0,00001, 0,001, 0,01, 0,1 | 6 regularizace vah |

`"passthrough"` znamená „tento krok vynech“. `PolynomialFeatures(..., interaction_only=True)` přidá součiny všech dvojic vstupů
(28 nových příznaků). Dvojité podtržítko odděluje název kroku a jeho parametr: `model__eta0` je parametr `eta0` kroku `model`.

### 6.1 Ruční ladění na pevné validační části

**Cíl / výstup:** `tune_base` – nenaučený řetězec s kroky `scale`, `features` a `model`; tabulka `manual_results`
s alespoň šesti variantami a jejich validačním AUC; slovník `manual_best_params` nejlepší varianty;
graf validačního AUC v závislosti na `eta0` se škálováním a bez něj.

1. `tune_base` = `Pipeline([("scale", StandardScaler()), ("features", "passthrough"), ("model", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, alpha=1e-4, random_state=SEED))])`.
   Chová se jako `logistic_pipe`; krok `features` jen dělá místo pro ladění.
2. **Před výpočtem** si zapište odhad: která změna nejvíc pomůže a která nejvíc uškodí?
3. Připravte seznam `manual_variants` s alespoň šesti slovníky parametrů z tabulky výše. Každý slovník mění **jednu** věc.
4. Pro každou variantu: `clone(tune_base)` → `set_params(**varianta)` → `fit` na trénovacích datech → validační AUC z `decision_function`.
   Výsledky uložte do `manual_results` seřazené od nejlepší a nejlepší slovník do `manual_best_params`.
5. Vyzkoušejte jednu **kombinaci** dvou nejlepších změn a přidejte ji do tabulky.
6. Graf: pro `eta0` 0,0001, 0,001, 0,01, 0,1, 1 spočtěte validační AUC se `StandardScaler()` a s `"passthrough"`
   a obě čáry nakreslete do jednoho grafu s logaritmickou osou x.

**Použijte:** `Pipeline`, `clone`, `.set_params(**slovník)`, `.decision_function`, `roc_auc_score`, cyklus `for`,
`pd.DataFrame(seznam_slovníků)`, `sort_values`, `plt.semilogx`.

<details><summary>Nápověda</summary>

Skóre pro AUC je `model.decision_function(X_val)`; existuje pro obě ztráty (perceptron pravděpodobnosti nemá).
Zápis `set_params(**{"model__eta0": 0.1})` je totéž co `set_params(model__eta0=0.1)`; prázdný slovník `{}` ponechá základní nastavení.
Objekty jako `PolynomialFeatures(...)` vytvářejte v každém slovníku znovu. Pomocná funkce `validate(params)`, která vrátí AUC
jedné varianty, zkrátí i kód pro graf.

</details>

**Zamyslete se:** Která změna pomohla a která uškodila? Odpovídá to vašemu odhadu a grafům z edukační části (oddíly 4, 5 a 10)?
Jak velký rozdíl AUC na 40 validačních vadných výliscích byste považovali za skutečný, a ne za náhodu?

### Ukázka: několik variant na pevné validační části

In [ ]:
demo_base = Pipeline([("scale", StandardScaler()), ("features", "passthrough"),  # Stejné tři kroky jako tune_base …
                      ("model", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.05, random_state=7))])  # … s konstantní rychlostí učení.
demo_variants = [{}, {"model__eta0": 0.5}, {"model__alpha": 0.05}]  # Prázdný slovník = základní nastavení.
demo_rows = []  # Výsledky variant.
for demo_params in demo_variants:  # Jedna varianta po druhé.
    demo_model = clone(demo_base).set_params(**demo_params)  # Kopie základního řetězce s jednou změnou.
    demo_model.fit(demo_X_train, demo_y_train)  # Stejná trénovací data pro všechny varianty.
    demo_auc = roc_auc_score(demo_y_val, demo_model.decision_function(demo_X_val))  # AUC ze skóre.
    demo_rows.append({"varianta": str(demo_params), "validační AUC": demo_auc})  # Jeden řádek tabulky.
display(pd.DataFrame(demo_rows).sort_values("validační AUC", ascending=False))  # Nejlepší varianta nahoře.

**Vlastní kód:**

### 6.2 Mřížkové hledání s křížovou validací

**Cíl / výstup:** `search` – naučený `GridSearchCV` nad `tune_base`; tabulka `grid_results` s deseti nejlepšími kombinacemi,
jejich průměrným AUC a směrodatnou odchylkou; graf nejlepšího průměrného AUC pro každou ztrátu v závislosti na `eta0`;
`manual_cv` – průměrné AUC vaší ruční volby ze stejné křížové validace.

1. Slovník `param_grid` s pěti parametry z tabulky v kroku 6: 2 · 2 · 2 · 5 · 4 = 160 kombinací.
2. `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)`.
3. `search = GridSearchCV(tune_base, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)` naučený na **vývojových datech**
   `X_dev`, `y_dev`. Testovací data zůstávají stranou. Výpočet trvá několik až desítky sekund.
4. Z `search.cv_results_` sestavte `grid_results`: sloupce `params`, `mean_test_score`, `std_test_score`, deset nejlepších řádků.
   Vypište `search.best_params_` a `search.best_score_`.
5. Graf: pro každou ztrátu nejlepší průměrné AUC (přes ostatní parametry) v závislosti na `eta0`.
6. Ohodnoťte `manual_best_params` z 6.1 stejnou křížovou validací (`cross_val_score` na `X_dev`, `y_dev`) a průměr uložte do `manual_cv`.
7. Ohodnoťte `search.best_params_` s `model__class_weight="balanced"` a bez něj stejnou křížovou validací. Změní se AUC? Proč tak málo?

**Použijte:** `GridSearchCV`, `StratifiedKFold`, `cross_val_score`, `search.best_params_`, `search.best_score_`,
`pd.DataFrame(search.cv_results_)`, `sort_values`, `groupby`.

<details><summary>Nápověda</summary>

Klíče `param_grid` jsou stejné názvy jako v `set_params`; hodnoty jsou seznamy. `cv_results_` obsahuje pro každý parametr sloupec
`param_<název>`, např. `param_model__eta0`. Hodnoty, které jsou objekty (škálovače, kroky příznaků), převeďte před seskupením na text
pomocí `.astype(str)`. Graf lze sestavit jako `tabulka.groupby(["eta0", "loss"])["score"].max().unstack().plot(marker="o", logx=True)`.

</details>

**Zamyslete se:** Odpovídá nejlepší kombinace vaší ruční volbě? Je rozdíl mezi pěti nejlepšími kombinacemi větší než jejich
směrodatná odchylka? Na kterém parametru záleží nejvíc a na kterém skoro vůbec? Odpovídá to edukační části?

### Ukázka: malá mřížka na ilustračních datech

In [ ]:
demo_grid = {"model__alpha": [0.0001, 0.05], "model__eta0": [0.005, 0.05]}  # 2 · 2 = 4 kombinace.
demo_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)  # Tři části pro malá data.
demo_search = GridSearchCV(demo_base, demo_grid, scoring="roc_auc", cv=demo_cv).fit(demo_X_dev, demo_y_dev)  # Každá kombinace na každé části.
demo_table = pd.DataFrame(demo_search.cv_results_)[["params", "mean_test_score", "std_test_score"]]  # Průměr a rozptyl přes části.
display(demo_table.sort_values("mean_test_score", ascending=False))  # Nejlepší kombinace nahoře.
print(demo_search.best_params_, round(demo_search.best_score_, 3))  # Vítěz a jeho průměrné AUC.
print("průměrné AUC jednoho nastavení:", cross_val_score(demo_base, demo_X_dev, demo_y_dev, scoring="roc_auc", cv=demo_cv).mean().round(3))  # cross_val_score pro jedno nastavení.

**Vlastní kód:**

### 6.3 Rozhodovací práh podle ceny chyb

**Cíl / výstup:** `chosen_params` – parametry výsledného modelu; `dev_proba` – pravděpodobnosti vady pro všechny vývojové
výlisky z modelů, které je při učení neviděly; tabulka `threshold_costs` s celkovou cenou pro prahy 0,01, 0,02, …, 0,99; nejlevnější práh
`chosen_threshold`; graf ceny (a přesnosti a úplnosti) v závislosti na prahu.

1. Práh potřebuje **pravděpodobnosti**, výsledný model proto musí být logistický neuron. `chosen_params` = `search.best_params_`,
   pokud je jeho `model__loss` rovno `"log_loss"`; jinak vezměte nejlepší řádek s `"log_loss"` ze všech kombinací, `pd.DataFrame(search.cv_results_)`.
2. `dev_proba = cross_val_predict(clone(tune_base).set_params(**chosen_params), X_dev, y_dev, cv=cv, method="predict_proba")[:, 1]`.
   Každý výlisek dostane pravděpodobnost od modelu, který ho při učení neviděl.
3. Pro každý práh t spočtěte předpovědi `dev_proba >= t` a jejich celkovou cenu pomocí `evaluate`. Tabulku uložte do `threshold_costs`
   a nejlevnější práh do `chosen_threshold`.
4. Porovnejte `chosen_threshold` s teoretickou hodnotou C_FP / (C_FP + C_FN) = 1/11 ≈ 0,091 a s cenou při 0,5.
5. Nakreslete cenu v závislosti na prahu; do druhého grafu přesnost a úplnost v závislosti na prahu.
6. Zopakujte kroky 2 a 3 s `model__class_weight="balanced"` přidaným do kopie `chosen_params` (samotné `chosen_params` neměňte).
   Kde je nejlevnější práh teď a proč? Je minimální cena nižší?

**Použijte:** `cross_val_predict`, cyklus `for` přes `np.round(np.arange(0.01, 1.0, 0.01), 2)`, `evaluate`, `pd.DataFrame`, `idxmin`, `plt.plot`.

<details><summary>Nápověda</summary>

Nejlepší kombinace s danou ztrátou: `tabulka = pd.DataFrame(search.cv_results_)`, potom `tabulka[tabulka["param_model__loss"] == "log_loss"].sort_values("mean_test_score").iloc[-1]["params"]`.
Pravděpodobnosti mimo učení pocházejí z pěti modelů, každý naučený na čtyřech pětinách `X_dev`; práh se proto volí bez testovacích dat
a bez optimismu předpovědí na trénovacích datech. Sbírejte řádky `{"threshold": t, **evaluate(y_dev, dev_proba >= t)}` a z nich sestavte tabulku.

</details>

**Zamyslete se:** Jak daleko je nejlevnější práh od 0,5? Kolik vad při něm přehlédnete a kolik dobrých výlisků pošlete na kontrolu?
Proč zde teoretická hodnota 1/11 platí jen přibližně?

### Ukázka: cena pro několik prahů

In [ ]:
demo_dev_proba = cross_val_predict(demo_base, demo_X_dev, demo_y_dev, cv=demo_cv, method="predict_proba")[:, 1]  # Pravděpodobnosti z modelů, které vzorek neviděly.
demo_rows = [{"threshold": t, **evaluate(demo_y_dev, demo_dev_proba >= t)} for t in (0.05, 0.1, 0.3, 0.5)]  # Jeden řádek na práh.
demo_costs = pd.DataFrame(demo_rows).set_index("threshold")  # Prahy jako index řádků.
display(demo_costs.round(3))  # Tabulka.
print("nejlevnější z těchto prahů:", demo_costs["cena"].idxmin())  # Řádek s nejnižší cenou.

**Vlastní kód:**

## 7. Zmrazení volby a závěrečný test

**Cíl / výstup:** `final_pipeline` – `tune_base` s `chosen_params`, naučený na všech vývojových datech;
testovací pravděpodobnosti `test_proba`; testovací rozhodnutí `test_pred` při `chosen_threshold`; tabulka `final_table`, která na testu srovnává
model při `chosen_threshold`, týž model při 0,5 a „vždy dobrý“; matice záměn.

Volba je zmrazená: parametry z 6.2, práh z 6.3. Vytvořte `final_pipeline` jako `clone(tune_base)` s `chosen_params`,
naučte ho na `X_dev`, `y_dev` (1500 výlisků) a teprve pak jednou předpovězte `X_test`.

**Použijte:** `clone`, `.set_params(**...)`, `.fit`, `.predict_proba`, `evaluate`, `ConfusionMatrixDisplay.from_predictions`.

<details><summary>Nápověda</summary>

Pro „vždy dobrý“ použijte `np.zeros(len(y_test), dtype=int)`. Testovací cíle slouží jen k vyhodnocení.

</details>

**Po zobrazení testovacích výsledků už následuje jen interpretace.** Změna prahu nebo parametrů podle testovacích výsledků
by z testovacích dat udělala další validační data.

**Zamyslete se:** Kolik vad model na testu přehlédl a kolik dobrých výlisků poslal na kontrolu?
Kolik ušetří zvolený práh oproti 0,5 a oproti žádnému modelu? Je testovací AUC blízko `search.best_score_`?

### Ukázka: nové naučení zmrazené volby a jeden test

In [ ]:
demo_final = clone(demo_base).set_params(**demo_search.best_params_).fit(demo_X_dev, demo_y_dev)  # Zmrazené nastavení, všechna vývojová data.
demo_test_proba = demo_final.predict_proba(demo_X_test)[:, 1]  # Jediná předpověď testovacích dat.
display(pd.DataFrame([evaluate(demo_y_test, demo_test_proba >= 0.1)], index=["práh 0,1"]).round(3))  # Vyhodnoceno jednou.

**Vlastní kód:**